In [22]:
import json
import sys
import math
import os
os.environ["CALORIES_DEBUG"] = 'true'

In [23]:
from pathlib import Path
current_dir = Path().resolve()
parent_dir = current_dir.parent
sys.path.append(str(parent_dir))
# from calories import *
#from calories import pandolfCalories, lcdaCalories, minimumMechanicsCalories, calorieEnsemble, load_sample_data, models

In [24]:
print(os.environ.get("CALORIES_DEBUG", "nope"))
# dir()

true


In [25]:
with open('../data/walk_31-i-m-on-unifi.json', 'r') as file:
    contents = file.read()
    walk_31 = json.loads(contents)

coords = walk_31["features"][0]["geometry"]["coordinates"]
print(coords[1])

[-122.20907615758304, 37.82453156141039, 87.28833691863991, 178.11343436688185, 22.783305624032607, 1784926645013]


In [27]:
def degs2Rads(degs: float) -> float:
    return degs * math.pi/180

def rads2Degs(rads: float) -> float:
    return rads * 180 / math.pi

rads2Degs(degs2Rads(180))

180.0

In [40]:
def crossTrackDistance(point: list(float), segment: list(list(float))) -> float:
    # Convert all longitudes and latitudes from degrees to radians
    point_lon, point_lat = math.radians(point[0]), math.radians(point[1])
    start_lon, start_lat = math.radians(segment[0][0]), math.radians(segment[0][1])
    end_lon, end_lat = math.radians(segment[1][0]), math.radians(segment[1][1])

    # Calculate the angular distance from start to point
    # Ensure the argument is within the domain of acos
    acos_argument = math.sin(start_lat) * math.sin(point_lat) + math.cos(start_lat) * math.cos(point_lat) * math.cos(point_lon - start_lon)
    acos_argument = max(-1, min(1, acos_argument))  # Clamp the argument between -1 and 1
    delta_sigma = math.acos(acos_argument)

    # Calculate the bearing from start to point and start to end
    theta_point = math.atan2(math.sin(point_lon - start_lon) * math.cos(point_lat),
                             math.cos(start_lat) * math.sin(point_lat) - math.sin(start_lat) * math.cos(point_lat) * math.cos(point_lon - start_lon))
    theta_end = math.atan2(math.sin(end_lon - start_lon) * math.cos(end_lat),
                           math.cos(start_lat) * math.sin(end_lat) - math.sin(start_lat) * math.cos(end_lat) * math.cos(end_lon - start_lon))

    # Calculate the cross track distance
    cross_track_dist = math.asin(math.sin(delta_sigma) * math.sin(theta_point - theta_end))
    # Convert cross track distance to kilometers by multiplying by the Earth's radius (6371 km)
    cross_track_dist = cross_track_dist * 6371
    return cross_track_dist


kbjc_runways = {
    "30R/12L": {
        "Runway 12L": {
            "Latitude": 39.91529286666667,
            "Longitude": -105.12841313333334
        },
        "Runway 30R": {
            "Latitude": 39.901373883333335,
            "Longitude": -105.10191808333333
        }
    }
}
test_locations = {
	"kbjc_tower": {
        "lat": 39.9064,
		"lon": -105.1208
	},  
}
kbjc_runway_30R_start = (kbjc_runways["30R/12L"]["Runway 30R"]["Longitude"], kbjc_runways["30R/12L"]["Runway 30R"]["Latitude"])
kbjc_runway_30R_end = (kbjc_runways["30R/12L"]["Runway 12L"]["Longitude"], kbjc_runways["30R/12L"]["Runway 12L"]["Latitude"])
kbjc_tower = (test_locations["kbjc_tower"]["lon"], test_locations["kbjc_tower"]["lat"])
def test_cross_track_distance():
    print(f"start lon: {kbjc_runway_30R_start[0]}, start lat: {kbjc_runway_30R_start[1]}")
    print(f"end lon: {kbjc_runway_30R_end[0]}, end lat: {kbjc_runway_30R_end[1]}")
    print(f"tower lon: {kbjc_tower[0]}, tower lat: {kbjc_tower[1]}")
    dist = crossTrackDistance(kbjc_tower, (kbjc_runway_30R_start, kbjc_runway_30R_end))
    print(f"cross track distance: {dist}")
test_cross_track_distance()

p1 = (-122.20130, 37.82531)
p2 = (-122.20263, 37.82850)
seg = list((coords[10], coords[int(len(coords)/2)]))
print(f'p1 {crossTrackDistance(p1, seg)}')
print(f'p2 {crossTrackDistance(p2, seg)}')

start lon: -105.10191808333333, start lat: 39.901373883333335
end lon: -105.12841313333334, end lat: 39.91529286666667
tower lon: -105.1208, tower lat: 39.9064
cross track distance: -0.44905534866747987
p1 0.03582144825558009
p2 -0.3331511204262821


In [ ]:
def ramerDouglasPeucker(points: list(list(float)), X: list(float), epsilon: int) -> list(list(float)):
    """A cartographic generalization algorithm to reduce the number of points in a 
       curve, composed of line segments, into a similar curve consisting of fewer points.

    Args:
        points (list(list(float)): A list of gps coordinate arrays.
        X: list(float): A gps coordinate array, in the same format as elements in points list.
        epsilon: float: A distance threshold value > 0.

    Returns:
        list(list(float)): A new array of gps points, of equal length as points, or fewer.
    """
    